In [ ]:
import sys
sys.path.insert(0, r"C:\Users\Admin\Desktop\ERP_System\ERP_System 3.0")

from __future__ import annotations

import logging

import pandas as pd

from erp_system.contracts import TABLE_CONTRACTS, validate_output_table
from erp_system.ingest.io_ops import (
    save_not_assigned_so,
    write_final_sales_order_to_gsheet,
    write_to_db,
)
from erp_system.ingest.sources import (
    extract_inputs,
    fetch_pdf_orders_df_from_DB,
    fetch_word_files_df,
    validate_input_tables,
)
from erp_system.ledger.atp import build_atp_view
from erp_system.ledger.events import _order_events, build_events, expand_nav_preinstalled
from erp_system.ledger.ledger import build_ledger_from_events
from erp_system.runtime.config import (
    DB_SCHEMA,
    TBL_INVENTORY,
    TBL_ITEM_ATP,
    TBL_ITEM_SUMMARY,
    TBL_LEDGER,
    TBL_POD,
    TBL_SALES_ORDER,
    TBL_Shipping,
    TBL_STRUCTURED,
)
from erp_system.runtime.policies import (
    GOOGLE_SHEET_SPREADSHEET,
    GOOGLE_SHEET_WORKSHEET,
    NOT_ASSIGNED_SO_EXPORT_PATH,
    WORD_FILE_API_URLS,
)
from erp_system.transform.inventory import add_onhand_minus_wip, build_wip_lookup, transform_inventory
from erp_system.transform.pod import enrich_pod_with_shipping_audit, transform_pod
from erp_system.transform.sales_order import transform_sales_order
from erp_system.transform.shipping import transform_shipping
from erp_system.transform.structured import build_structured_df, prepare_erp_view

In [9]:
from __future__ import annotations

import json

import pandas as pd
import requests

from erp_system.runtime.config import POD_FILE, SALES_ORDER_FILE, SHIPPING_SCHEDULE_FILE, WAREHOUSE_INV_FILE
from erp_system.runtime.db_config import get_engine
from erp_system.transform.sales_order import normalize_wo_number


def fetch_pdf_orders_df_from_DB() -> pd.DataFrame:
    eng = get_engine()
    rows = pd.read_sql("SELECT order_id, extracted_data FROM public.pdf_file_log", eng)

    def rows_from_json(extracted_data, order_id=""):
        if isinstance(extracted_data, str):
            try:
                extracted_data = json.loads(extracted_data)
            except Exception:
                extracted_data = {}
        data = extracted_data or {}
        wo = normalize_wo_number(data.get("wo", order_id))
        consigned = data.get("Consigned")
        if consigned is None:
            consigned = data.get("consigned")
        consigned = bool(consigned) if consigned is not None else False
        items = data.get("items") or []
        if not items:
            return [{"WO": wo, "Product Number": "", "Consigned": consigned}]
        out = []
        for it in items:
            pn = it.get("product_number") or it.get("part_number") or it.get("product") or it.get("part") or ""
            out.append({"WO": wo, "Product Number": pn, "Consigned": consigned})
        return out

    all_rows = []
    for _, r in rows.iterrows():
        all_rows.extend(rows_from_json(r.get("extracted_data"), r.get("order_id")))
    return pd.DataFrame(all_rows, columns=["WO", "Product Number", "Consigned"])

pdf_so = fetch_pdf_orders_df_from_DB()
pdf_so.loc[pdf_so["WO"] == "SO-20260264"]

,WO,Product Number,Consigned
1604,SO-20260264,NRU-162S-AWP,False
1605,SO-20260264,GC-Jetson-NX8G-Orin-Nvidia,False
1606,SO-20260264,M.242-SSD-128GB-PCIe34-TLC5WT-TD,False
1607,SO-20260264,RPnl-2Ant-NRU-162S-AWP,False
1608,SO-20260264,M.2-LTE-7455,False
1609,SO-20260264,Risr-M2B-mPCIe-SIMslot,False
1610,SO-20260264,Cbl-MHF4-SMAF-15CM,False
1611,SO-20260264,DtC-SMA-WP,False
1612,SO-20260264,Cblkit-NRU-162S-AWP,False
1613,SO-20260264,PA-120W-OW,False
